# LAB | Intro to Machine Learning

**Load the data**

In this challenge, we will be working with Spaceship Titanic data. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In [55]:
#import libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [56]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


**Check the shape of your data**

In [57]:
spaceship.shape

(8693, 14)

**Check for data types**

In [58]:
spaceship.dtypes

PassengerId      object
HomePlanet       object
CryoSleep        object
Cabin            object
Destination      object
Age             float64
VIP              object
RoomService     float64
FoodCourt       float64
ShoppingMall    float64
Spa             float64
VRDeck          float64
Name             object
Transported        bool
dtype: object

**Check for missing values**

In [59]:
spaceship.isna().sum()



PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64

There are multiple strategies to handle missing data

- Removing all rows or all columns containing missing data.
- Filling all missing values with a value (mean in continouos or mode in categorical for example).
- Filling all missing values with an algorithm.

For this exercise, because we have such low amount of null values, we will drop rows containing any missing value. 

In [60]:
# for now, those null values in numeric variables has no actual meaning , i will impute null values with their median 
numeric_cols = spaceship.select_dtypes(include='number').columns
print(numeric_cols)
spaceship['Age_clean'] = spaceship['Age'].fillna(spaceship['Age'].median())
spaceship['RoomService_clean'] = spaceship['RoomService'].fillna(spaceship['RoomService'].median())
spaceship['FoodCourt_clean'] = spaceship['FoodCourt'].fillna(spaceship['FoodCourt'].median())
spaceship['ShoppingMall_clean'] = spaceship['ShoppingMall'].fillna(spaceship['ShoppingMall'].median())
spaceship['Spa_clean'] = spaceship['Spa'].fillna(spaceship['Spa'].median())
spaceship['VRDeck_clean'] = spaceship['VRDeck'].fillna(spaceship['VRDeck'].median())

# for now, those null values in categorical variables has also no actual meaning, imput null values with mode
categorical_cols = spaceship.select_dtypes(include='object').columns
categorical_cols = categorical_cols.difference(['PassengerId','Name'])
print(categorical_cols)
spaceship['Cabin_clean'] = spaceship['Cabin'].fillna(spaceship['Cabin'].mode()[0])
spaceship['CryoSleep_clean'] = spaceship['CryoSleep'].fillna(spaceship['CryoSleep'].mode()[0])
spaceship['Destination_clean'] = spaceship['Destination'].fillna(spaceship['Destination'].mode()[0])
spaceship['HomePlanet_clean'] = spaceship['HomePlanet'].fillna(spaceship['HomePlanet'].mode()[0])
spaceship['VIP_clean'] = spaceship['VIP'].fillna(spaceship['VIP'].mode()[0])


cols_include = [col for col in spaceship.columns if col.endswith('clean')]
cols_include = cols_include + ['PassengerId','Name','Transported']
cols_include
spaceship = spaceship.copy()
spaceship = spaceship[cols_include]

Index(['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck'], dtype='object')
Index(['Cabin', 'CryoSleep', 'Destination', 'HomePlanet', 'VIP'], dtype='object')


/var/folders/zd/t_pj2nzd2_x4z46k814jg73r0000gn/T/ipykernel_21160/286292101.py:16: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  spaceship['CryoSleep_clean'] = spaceship['CryoSleep'].fillna(spaceship['CryoSleep'].mode()[0])
/var/folders/zd/t_pj2nzd2_x4z46k814jg73r0000gn/T/ipykernel_21160/286292101.py:19: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  spaceship['VIP_clean'] = spaceship['VIP'].fillna(spaceship['VIP'].mode()[0])


In [61]:
spaceship.isna().sum()

spaceship.dtypes

# CryoSleep_clean, VIP_clean becomes bool type 
# for EDA/stats, they are categorical variable 
# for machine learning, leave it as bool is fine, it will be converted to 0/1 (numeric) automatically 


Age_clean             float64
RoomService_clean     float64
FoodCourt_clean       float64
ShoppingMall_clean    float64
Spa_clean             float64
VRDeck_clean          float64
Cabin_clean            object
CryoSleep_clean          bool
Destination_clean      object
HomePlanet_clean       object
VIP_clean                bool
PassengerId            object
Name                   object
Transported              bool
dtype: object

**KNN**

K Nearest Neighbors is a distance based algorithm, and requeries all **input data to be numerical.**

Let's only select numerical columns as our features.

In [74]:
features_col = spaceship.columns.difference(['Cabin_clean','Destination_clean','HomePlanet_clean','PassengerId','Name','Transported'])

feature_df = spaceship[features_col]
feature_df



,Age_clean,CryoSleep_clean,FoodCourt_clean,RoomService_clean,ShoppingMall_clean,Spa_clean,VIP_clean,VRDeck_clean
0,39.0,False,0.0,0.0,0.0,0.0,False,0.0
1,24.0,False,9.0,109.0,25.0,549.0,False,44.0
2,58.0,False,3576.0,43.0,0.0,6715.0,True,49.0
3,33.0,False,1283.0,0.0,371.0,3329.0,False,193.0
4,16.0,False,70.0,303.0,151.0,565.0,False,2.0
...,...,...,...,...,...,...,...,...
8688,41.0,False,6819.0,0.0,0.0,1643.0,True,74.0
8689,18.0,True,0.0,0.0,0.0,0.0,False,0.0
8690,26.0,False,0.0,0.0,1872.0,1.0,False,0.0
8691,32.0,False,1049.0,0.0,0.0,353.0,False,3235.0


And also lets define our target.

In [73]:
target_df = spaceship['Transported']
target_df

0       False
1        True
2       False
3       False
4        True
        ...  
8688    False
8689    False
8690     True
8691    False
8692     True
Name: Transported, Length: 8693, dtype: bool

**Train Test Split**

Now that we have split the data into **features** and **target** variables and imported the **train_test_split** function, split X and y into X_train, X_test, y_train, and y_test. 80% of the data should be in the training set and 20% in the test set.

In [79]:

X_train, X_test, y_train, y_test = train_test_split(feature_df, target_df, test_size=0.20, random_state=0)


**Model Selection**

In this exercise we will be using **KNN** as our predictive model.

You need to choose between **Classificator** or **Regressor**. Take into consideration target variable to decide.

Initialize a KNN instance without setting any hyperparameter.

In [103]:
# Transported is naturally a yes/no outcome, choose classfication

from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier() # default k=5

Fit the model to your data.

In [102]:
knn.fit(X_train,y_train)

y_pred = knn.predict(X_test)

Evaluate your model.

In [101]:
# check if classes are balanced in test data, is balanced 
y_test.value_counts() 

from sklearn.metrics import accuracy_score,precision_score,recall_score,confusion_matrix

# confusing_matrix 
# 618 true negative - acutal false, predict false
# 245 false positive - actual false, predict true
# 204 false negative - actual true, predict false
# 672. true positive - actual true, predict true

cm = confusion_matrix(y_test,y_pred,labels=[False, True])
print(cm)

# Accuracy 
print("Accuracy:",accuracy_score(y_test,y_pred))

# Precision = of predicted "True", how many were actually "True". = 672/(672+245)
print("Precision:", precision_score(y_test, y_pred))

# Recall = of actual "True", how many did we catch.
print("Recall   :", recall_score(y_test, y_pred))


[[618 245]
 [204 672]]
Accuracy: 0.7418056354226567
Precision: 0.732824427480916
Recall   : 0.7671232876712328


**Congratulations, you have just developed your first Machine Learning model!**